In [4]:
import requests
from bs4 import BeautifulSoup

url = "https://biomassmagazine.com/tag/advanced-biofuels/4"  # or whatever page you're scraping

# Add a user-agent + timeout for reliability
resp = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=20
)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")

# Extract article URLs using the CSS classes you identified
article_links = soup.select("p.chakra-text.css-bnnz7e a.chakra-link.css-spn4bz")

# Extract the href attributes and convert to absolute URLs
base_url = "https://biomassmagazine.com"
article_urls = []

for link in article_links:
    href = link.get("href")
    if href:
        # Convert relative URLs to absolute URLs
        if href.startswith("/"):
            full_url = base_url + href
        else:
            full_url = href
        article_urls.append(full_url)

# Print the extracted URLs
print(f"Found {len(article_urls)} article URLs:")
for i, url in enumerate(article_urls, 1):
    print(f"{i}. {url}")

# Or just get the list
print("\nArticle URLs list:")
print(article_urls)

Found 30 article URLs:
1. https://biomassmagazine.com/articles/epri-tva-complete-worlds-largest-renewable-diesel-powered-combustion-turbine-demonstration
2. https://biomassmagazine.com/articles/clean-fuels-elects-governing-board-members
3. https://biomassmagazine.com/articles/ny-waterway-completes-renewable-diesel-trial-moves-forward-with-significant-conversion-to-cleaner-energy-source
4. https://biomassmagazine.com/articles/doe-announces-202-million-in-projects-to-advance-development-of-mixed-algae-for-biofuels-and-bioproducts
5. https://biomassmagazine.com/articles/clean-fuels-soy-growers-urge-congress-to-extend-biodiesel-tax-credit
6. https://biomassmagazine.com/articles/eia-maintains-forecasts-for-biodiesel-renewable-diesel-and-saf-production
7. https://biomassmagazine.com/articles/carb-votes-to-adopt-lcfs-updates
8. https://biomassmagazine.com/articles/marathon-martinez-biorefinery-to-reach-full-capacity-by-year-end
9. https://biomassmagazine.com/articles/aemetis-announces-support

In [3]:
import requests
from bs4 import BeautifulSoup
import json
import html

def extract_biomass_article_paragraphs(url):
    """Extract paragraphs from a Biomass Magazine article"""

    # Add a user-agent + timeout for reliability
    resp = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=20
    )
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")

    # Find the script tag containing the article data
    script_tag = soup.find("script", {"id": "__NEXT_DATA__"})

    if script_tag:
        try:
            # Parse the JSON data
            json_data = json.loads(script_tag.string)

            # Extract article information
            article = json_data.get("props", {}).get("pageProps", {}).get("article", {})

            title = article.get("title", "No title found")
            body_html = article.get("body", "")
            published_at = article.get("publishedAt")
            author = article.get("author", {}).get("name", "Unknown")
            summary = article.get("summary", "")

            # Decode HTML entities and parse the body content
            decoded_body = html.unescape(body_html)
            body_soup = BeautifulSoup(decoded_body, "html.parser")

            # Extract paragraphs as p1, p2, p3...
            paragraph_nodes = body_soup.find_all("p")
            paragraphs = {
                f"p{idx+1}": p.get_text(" ", strip=True)
                for idx, p in enumerate(paragraph_nodes)
                if p.get_text(strip=True)
            }

            # Build the article data structure
            article_data = {
                "title": title,
                "paragraphs": [paragraphs],  # ✅ wrap dict in a list for consistency
                "meta": {
                    "date": published_at,
                    "url": url,
                    "author": author,
                    "summary": summary,
                    "source": "biomass_magazine"
                },
            }

            return article_data

        except json.JSONDecodeError as e:
            print(f"Error parsing JSON: {e}")
            return None
        except Exception as e:
            print(f"Error extracting article: {e}")
            return None
    else:
        print("No article data found in script tag")
        return None


# --- Run Test ---
if __name__ == "__main__":
    test_url = "https://biomassmagazine.com/articles/phillips-66-expects-renewable-diesel-margins-to-improve"
    article_data = extract_biomass_article_paragraphs(test_url)

    if article_data:
        # Pretty-print JSON for readability
        print(json.dumps(article_data, indent=4, ensure_ascii=False))
    else:
        print("❌ Failed to extract article data")


{
    "title": "Phillips 66 expects renewable diesel margins to improve",
    "paragraphs": [
        {
            "p1": "Phillips 66 on Oct. 29 reported that its renewable fuels segment was impacted by lower margins during the third quarter of this year, but company officials expect margins to improve moving forward. The company also reported it began producing sustainable aviation fuel (SAF) in September.",
            "p2": "The company currently produces renewable fuels at its Rodeo Renewable Energy Complex, a biorefinery project has been under development since mid-2022 and reached full processing rates during the second quarter of 2024.",
            "p3": "Kevin Mitchell, chief financial officer at Phillips 66, reported that the Rodeo facility produced 44,000 barrels per day of renewable fuels during the third quarter. The biorefinery has a nameplate capacity of approximately 50,000 barrels per day (800 MMgy).",
            "p4": "Brian Mandell, executive vice president of mark